# Entrenamiento de Redes Neuronales de Grafos


Este estregable tiene por objetivo entrenar redes neuronales de grafos GNN (Graph Neuronal Networks) para predecir transacciones fraudulentas de Bitcoin en el dataset de Elliptic.

## Grafo de Elliptic


Antes que nada, se debe cargar el grafo y aplicar transformaciones para usarlo de input en los modelos.

Para ello se recurre al grafo cargado en la librería PyTorch Geomettric (PyG) donde se encuentra como objeto del tipo 'Data' que contiene toda la info del grafo.

Este tipo objeto de datos posee distintos atributos, entre ellos: 
1) edge_index contiene información sobre la conectividad del grafo; es decir, una tupla de índices de nodo de origen y destino para cada arista.
2) características de los nodos como 'x' (cada uno de los nodos tiene asignado un vector de características de n dimensiones). 
3) las etiquetas de los nodos como 'y' (cada nodo tiene asignado exactamente una clase).
4) También existe un atributo adicional llamado 'train_mask', que describe para qué nodos ya conocemos sus asignaciones de comunidad.


In [18]:
from torch_geometric.datasets import EllipticBitcoinDataset

# Ruta donde se van a guardar los archivos
root = "data/elliptic"

# Descargar / procesar el dataset
dataset = EllipticBitcoinDataset(root=root)

# Extraer el grafo (es único, por eso dataset[0])
data = dataset[0]

print(data)
print(data.keys())   # atributos disponibles
print(data.x.shape)  # matriz de features
print(data.edge_index.shape)  # aristas
print(data.y.shape)  # etiquetas

Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], test_mask=[203769])
['edge_index', 'y', 'x', 'train_mask', 'test_mask']
torch.Size([203769, 165])
torch.Size([2, 234355])
torch.Size([203769])


In [19]:
# Ver distribución de etiquetas
import torch

print(torch.unique(data.y, return_counts=True))

(tensor([0, 1, 2]), tensor([ 42019,   4545, 157205]))


Se filtra aquellos nodos etiquetados, eliminando los desconocidos. Recordar que 0: lícito, 1: fraude, 2: desconocido

In [20]:
from torch_geometric.data import Data
from torch_geometric.utils import subgraph

# Índices de nodos con etiqueta conocida
mask = data.y != 2
node_idx = mask.nonzero(as_tuple=True)[0]

# Filtrar aristas y reindexar
edge_index, _ = subgraph(node_idx, data.edge_index, relabel_nodes=True, num_nodes=data.num_nodes)

# Crear un nuevo objeto Data
data_filtered = Data(
    x=data.x[node_idx],
    y=data.y[node_idx],
    edge_index=edge_index
)

print(data_filtered)
print(data_filtered.x.shape)   # nodos filtrados
print(data_filtered.y.bincount())  # cantidad de etiquetas 0 y 1


Data(x=[46564, 165], edge_index=[2, 36624], y=[46564])
torch.Size([46564, 165])
tensor([42019,  4545])


El dataset Elliptic está pensado para entrenar por ventanas temporales, usando la columna time_step. 

Por ende, se regeneran train_mask, val_mask y test_mask sobre este data_filtered usando los time_step originales:

In [21]:
# Primeras 5 filas completas de features
print(data.x[:5])

tensor([[-1.7147e-01, -1.8467e-01, -1.2014e+00, -1.2197e-01, -4.3875e-02,
         -1.1300e-01, -6.1584e-02, -1.6210e-01, -1.6793e-01, -4.9707e-02,
         -1.6440e-01, -2.8741e-02, -3.5391e-02, -4.2955e-02, -1.3282e-02,
         -5.7195e-02, -1.6961e-01, -1.7115e-01, -1.7447e-01, -1.3737e+00,
         -1.3715e+00, -1.3973e-01, -1.4891e-01, -8.0147e-02, -1.5566e-01,
         -1.0763e-02, -1.2107e-02, -1.3973e-01, -1.4891e-01, -8.0147e-02,
         -1.5566e-01, -1.0669e-02, -1.2005e-02, -2.4669e-02, -3.1272e-02,
         -2.3045e-02, -2.6215e-02,  1.4278e-03,  1.4826e-03, -2.2722e-01,
         -2.3937e-01, -7.5256e-02, -2.3495e-01,  3.7468e-02,  4.3444e-02,
         -2.2720e-01, -2.4324e-01, -9.7895e-02, -2.3590e-01,  3.6577e-02,
          4.2345e-02, -4.1401e-01, -4.8834e-01, -2.3255e-01, -4.6755e-01,
          4.8767e-02,  5.2956e-02, -3.9149e-02, -1.7290e-01, -1.6313e-01,
         -1.6093e-01, -1.3163e+00, -1.3154e+00, -3.9144e-02, -1.7288e-01,
         -1.6311e-01, -1.6093e-01, -1.

In [16]:
# Extraer columna de time_step desde X original
time_steps_all = data.x[:, 93].long() # la columna 93 es la de los time_step

# Filtrado con nodos válidos
time_steps = time_steps_all[node_idx]

print(time_steps.min().item(), time_steps.max().item())  # debería ir de 1 a 49

# Definir cortes temporales (lo habitual es:train = steps 1–35, val = steps 36–40, test = steps 41–49)

y = data_filtered.y
train_mask = ((time_steps >= 1) & (time_steps <= 35))
val_mask   = ((time_steps >= 36) & (time_steps <= 40))
test_mask  = ((time_steps >= 41) & (time_steps <= 49))

# Agregar las máscaras al objeto 
data_filtered.train_mask = train_mask
data_filtered.val_mask   = val_mask
data_filtered.test_mask  = test_mask

# Chequear tamaños
print("Train:", train_mask.sum().item())
print("Val:", val_mask.sum().item())
print("Test:", test_mask.sum().item())


0 46
Train: 1775
Val: 1
Test: 17


Ya está todo listo para empezar a entrenar modelos GNN.

## Graph Convolutional Network - GCNConv
